[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Favioleiva/diarizator/blob/main/notebooks/Diarizator_Community1_Colab.ipynb)

# Diarizator Community-1 — upload-first Colab

**Standard workflow:** add `HF_TOKEN` to Colab Secrets, select **Run all**, upload one supported file, and download the sanitized ZIP. Google Drive is not required. Select a GPU runtime first.

Community-1 is gated: accept the conditions on the [official model page](https://huggingface.co/pyannote/speaker-diarization-community-1), create a read token, add it as the `HF_TOKEN` secret, and enable notebook access. Never paste a token into a cell.


### Step 1 — Install and validate the dependency profile

Run the code cell below by clicking the **▶** button on its left.

On the first execution, Diarizator may install or update the validated Python dependency profile. If that happens, the cell will stop intentionally and display a message asking you to restart the Colab session.

When you see that message:

1. Select **Runtime → Restart session**.
2. After the restart, run **this same dependency cell again**.
3. Continue only when the cell finishes with:

`DEPENDENCY_PROFILE_OK`

The first stop is expected and does **not** indicate an installation failure.


In [ ]:
# Validated dependency profile and safe two-pass restart guard.
import importlib.metadata as md
import json
import subprocess
import sys
import urllib.request

OWNER = "Favioleiva"
REPOSITORY = "diarizator"
REF = "main"  # Replace with v0.1.0 only after the release is validated.

REPO_URL = f"https://github.com/{OWNER}/{REPOSITORY}.git"
RAW = f"https://raw.githubusercontent.com/{OWNER}/{REPOSITORY}/{REF}/colab"

EXPECTED = {
    "numpy": "2.2.2",
    "scipy": "1.16.3",
    "pandas": "2.2.3",
    "jedi": "0.19.2",
    "faster-whisper": "1.2.1",
    "pyannote.audio": "4.0.7",
    "pyannote.metrics": "4.1",
    "opentelemetry-api": "1.42.1",
    "opentelemetry-sdk": "1.42.1",
    "tokenizers": "0.23.1",
    "fsspec": "2025.12.0",
}

requirements = "/content/requirements-colab.txt"
constraints = "/content/constraints-colab.txt"


def setup_dependencies():

    for url, target in (
        (RAW + "/requirements-colab.txt", requirements),
        (RAW + "/constraints-colab.txt", constraints),
    ):
        urllib.request.urlretrieve(url, target)

    current = {}

    for package in EXPECTED:
        try:
            current[package] = md.version(package)
        except md.PackageNotFoundError:
            current[package] = None

    binary_probe = subprocess.run(
        [
            sys.executable,
            "-c",
            "import numpy, scipy; print(numpy.__version__, scipy.__version__)",
        ],
        capture_output=True,
        text=True,
    )

    needs_install = current != EXPECTED or binary_probe.returncode != 0

    if needs_install:

        subprocess.check_call(
            [
                sys.executable,
                "-m",
                "pip",
                "install",
                "--upgrade",
                "--force-reinstall",
                "numpy==2.2.2",
                "scipy==1.16.3",
            ]
        )

        subprocess.check_call(
            [
                sys.executable,
                "-m",
                "pip",
                "install",
                "-r",
                requirements,
                "-c",
                constraints,
            ]
        )

        subprocess.check_call(
            [
                sys.executable,
                "-m",
                "pip",
                "install",
                "--upgrade",
                "--force-reinstall",
                "tokenizers==0.23.1",
                "fsspec==2025.12.0",
            ]
        )

        subprocess.check_call(
            [
                sys.executable,
                "-m",
                "pip",
                "install",
                "--no-deps",
                f"diarizator @ git+{REPO_URL}@{REF}",
            ]
        )

        probe = subprocess.run(
            [
                sys.executable,
                "-c",
                (
                    "import numpy, scipy, pandas, tokenizers, fsspec, diarizator; "
                    "print('BINARY_IMPORTS_OK'); "
                    "print('numpy', numpy.__version__); "
                    "print('scipy', scipy.__version__); "
                    "print('pandas', pandas.__version__); "
                    "print('tokenizers', tokenizers.__version__); "
                    "print('fsspec', fsspec.__version__)"
                ),
            ],
            capture_output=True,
            text=True,
        )

        if probe.returncode != 0:
            raise RuntimeError(probe.stderr[-2000:])

        print(
            "\n"
            "✅ Dependencies installed and verified successfully.\n"
            "\n"
            "A Colab session restart is now required because core Python\n"
            "packages were updated.\n"
            "\n"
            "Next steps:\n"
            "  1. Select Runtime → Restart session.\n"
            "  2. After the restart, run THIS SAME CELL again.\n"
            "  3. Continue only when you see DEPENDENCY_PROFILE_OK.\n"
            "\n"
            "This is expected and is not an installation failure."
        )

        return False

    subprocess.check_call(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "--no-deps",
            f"diarizator @ git+{REPO_URL}@{REF}",
        ]
    )

    check = subprocess.run(
        [sys.executable, "-m", "pip", "check"],
        capture_output=True,
        text=True,
    )

    known = (
        "google-colab 1.0.0 has requirement pandas==2.2.2",
        "numba 0.60.0 has requirement numpy<2.1,>=1.22",
    )

    benign = {
        "No broken requirements found.",
    }

    unexpected = [
        line
        for line in check.stdout.splitlines()
        if line
        and line not in benign
        and not any(item in line for item in known)
    ]

    if unexpected:
        raise RuntimeError(
            "Unexpected dependency conflicts: "
            + json.dumps(unexpected)
        )

    print("✅ DEPENDENCY_PROFILE_OK")
    return True


DEPENDENCIES_READY = setup_dependencies()

## Optional configuration

The default is **Use bundled demonstration audio** with automatic speaker-count estimation. Change it to **"Upload my own audio"** for your own audio. Advanced users may choose exact or bounded counts. The bundled demonstration may use its known exact two-character structure; arbitrary uploads never silently use that setting.


In [ ]:
INPUT_CHOICE = "Use bundled demonstration audio"  # or: Upload my own audio
SPEAKER_COUNT_MODE = "estimated"  # estimated, exact, or bounded
NUM_SPEAKERS = None
MIN_SPEAKERS = None
MAX_SPEAKERS = None
DEMO_SPEAKER_COUNT_MODE = "exact"  # exact or estimated
if SPEAKER_COUNT_MODE not in {"estimated", "exact", "bounded"}: raise ValueError("Invalid speaker-count mode")
if SPEAKER_COUNT_MODE == "exact" and (not isinstance(NUM_SPEAKERS, int) or NUM_SPEAKERS < 1): raise ValueError("Exact mode requires NUM_SPEAKERS >= 1")
if SPEAKER_COUNT_MODE == "bounded" and (not isinstance(MIN_SPEAKERS, int) or not isinstance(MAX_SPEAKERS, int) or MIN_SPEAKERS < 1 or MAX_SPEAKERS <= MIN_SPEAKERS): raise ValueError("Bounded mode requires 1 <= MIN_SPEAKERS < MAX_SPEAKERS")


In [ ]:
# Secure authentication and Community-1 access preflight happen before ASR.
from google.colab import userdata
from diarizator.access import preflight_model_access
HF_TOKEN = userdata.get("HF_TOKEN")
access = preflight_model_access(token=HF_TOKEN)
print("Community-1 access:", access["status"])
if access["status"] not in {"MODEL_CACHED", "MODEL_ACCESS_CONFIRMED"}:
    raise RuntimeError("Community-1 preflight stopped safely: " + access["status"])


## Select one input

Supported formats include MP3, M4A/AAC, WAV, FLAC, OGG/OPUS, and audio tracks from common MP4, MOV, MKV, and WebM files. For faster browser uploads, a compressed format such as MP3 is recommended. M4A/AAC is also compact. WAV files are usually much larger. This is about upload size—not accuracy.


In [ ]:
from pathlib import Path
import shutil, subprocess, sys, tempfile
from google.colab import files
workspace = Path(tempfile.mkdtemp(prefix="diarizator-"))
if INPUT_CHOICE == "Upload my own audio":
    uploaded = files.upload()
    if len(uploaded) != 1: raise ValueError("Upload exactly one supported audio or video file.")
    filename, payload = next(iter(uploaded.items()))
    audio_path = workspace / Path(filename).name
    audio_path.write_bytes(payload)
    mode, number, lower, upper = SPEAKER_COUNT_MODE, NUM_SPEAKERS, MIN_SPEAKERS, MAX_SPEAKERS
elif INPUT_CHOICE == "Use bundled demonstration audio":
    source = "https://raw.githubusercontent.com/Favioleiva/diarizator/main/examples/controlled_demo/The_Measure_of_a_Good_Life.mp3"
    audio_path = workspace / "The_Measure_of_a_Good_Life.mp3"
    urllib.request.urlretrieve(source, audio_path)
    mode = DEMO_SPEAKER_COUNT_MODE
    number, lower, upper = (2 if mode == "exact" else None), None, None
else: raise ValueError("Unknown input choice")
from diarizator.media import inspect_media
print(json.dumps(inspect_media(audio_path), indent=2))


In [ ]:
from diarizator import diarize
from diarizator.bundle import create_bundle
result_dir = diarize(audio_path, workspace / "results", token=HF_TOKEN, speaker_count_mode=mode, num_speakers=number, min_speakers=lower, max_speakers=upper)
bundle_path = workspace / "diarizator-results.zip"
bundle = create_bundle([result_dir], bundle_path)
run = json.loads((result_dir / "run.json").read_text())
environment = json.loads((result_dir / "environment.json").read_text())
diarization = json.loads((result_dir / "diarization.json").read_text())
print(json.dumps({"zip_filename": bundle_path.name, "zip_path": str(bundle_path), "zip_size_bytes": bundle_path.stat().st_size, "zip_sha256": bundle["sha256"], "detected_gpu": environment["device"], "backend": run["config"]["diarization"]["backend"], "speaker_count_mode": run["speaker_count_mode"], "detected_speaker_count": diarization["estimated_speakers"], "run_status": run["status"]}, indent=2))
files.download(str(bundle_path))
